# Meta-Llama-3-8B — intrinsic evaluation

Scores `meta-llama/Meta-Llama-3-8B` on the same 500 parallel sentence pairs and
500 mixed-script sentences as the other checkpoints, and writes the same two
CSVs.

**Why this run exists.** SinLlama is a continual pre-training of Llama-3-8B with
a Sinhala-extended tokenizer. The pool already contains Llama-3.**1**-8B, so
comparing SinLlama against it would mix the Sinhala adaptation together with the
3 to 3.1 model update. Scoring the exact parent isolates the adaptation.

**Setup.** Accelerator: GPU T4 x2. Internet: on. Secret `HF_TOKEN` must be
attached, with access granted for the gated Llama-3 repository. Attach the
intrinsic evaluation dataset (`diverse_sentences.csv` and
`sinhala_mixed_dataset.csv`) as an input.

**Outputs to commit back to the repository**, under
`results/intrinsic_evaluation/Meta-Llama-3-8B/`:

* `Meta-Llama-3-8B_diverse_intrinsic.csv`
* `Meta-Llama-3-8B_mixed_intrinsic.csv`
* `pilot_intrinsic_results.csv`

The last cell also prints every pooled number, so the summary can be read off
the log if the CSVs are awkward to move.

Expected wall-clock time is well under an hour: 1,500 scored sequences, one
forward pass each.

Environment Setup and Authentication

In [1]:
import sys
import os

# Upgrade Hugging Face ecosystem safely
!{sys.executable} -m pip install --upgrade --quiet transformers accelerate huggingface_hub datasets

import torch
import math
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | Detected {device_count} GPUs.")

# Authenticatation
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Authenticated via Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = ""
    login(token=HF_TOKEN)
    print("Authenticated via explicit string.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 67.0 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 63.7 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
PyTorch 2.10.0+cu128 | Detected 2 GPUs.
Authenticated via Kaggle Secrets.


Evaluation Metrices

In [2]:
class IntrinsicEvaluator:
    def __init__(self, model, tokenizer, device="cuda:0"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def get_lengths(self, text: str):
        """Calculates text-intrinsic properties for normalization."""
        if not isinstance(text, str):
            text = str(text)
        num_chars = max(1, len(text))
        num_bytes = max(1, len(text.encode('utf-8')))
        num_words = max(1, len(text.strip().split()))
        return num_chars, num_bytes, num_words

    def evaluate_sequence(self, text: str):
        """Calculates raw NLL plus the raw counts needed to pool correctly across the
        whole corpus. bpb/bpc/bpw below are still returned for per-sentence inspection,
        but the CORPUS-LEVEL (headline) numbers must be computed by pooling nll and the
        *_count fields across all rows -- never by averaging these per-row ratios."""
        chars, bytes_count, words = self.get_lengths(text)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            seq_len = inputs["input_ids"].shape[1]
            n_scored = seq_len - 1          # tokens actually predicted -- needed for token PPL
            neg_log_likelihood = outputs.loss.item() * n_scored

        bpb = neg_log_likelihood / (bytes_count * math.log(2))
        bpc = neg_log_likelihood / (chars * math.log(2))
        bpw = neg_log_likelihood / (words * math.log(2))

        return {
            "nll": neg_log_likelihood,
            "bpb": bpb, "bpc": bpc, "bpw": bpw,
            "n_scored": n_scored,
            "n_chars": chars, "n_bytes": bytes_count, "n_words": words,
        }
print("IntrinsicEvaluator class defined.")

IntrinsicEvaluator class defined.


Standardized Data Loading & Preprocessing

In [3]:
import glob

diverse_sentences_path = "/kaggle/input/datasets/anon-owner-b/intrinsic/diverse_sentences.csv"
sinhala_mixed_path = "/kaggle/input/datasets/anon-owner-b/intrinsic/sinhala_mixed_dataset.csv"

def resolve_path(preferred_path, filename_pattern):
    """Use the hard-coded path if it exists in this session; otherwise search
    /kaggle/input for a matching file. Datasets can be attached under a
    different owner/slug, or the exact filename can shift between versions
    (e.g. 'diverse_sentences.csv' vs 'diverse_sentences (1).csv'), which
    silently breaks a hard-coded path across sessions."""
    if os.path.exists(preferred_path):
        return preferred_path
    matches = glob.glob(f"/kaggle/input/**/{filename_pattern}", recursive=True)
    if matches:
        print(f"Note: preferred path not found, using discovered path instead: {matches[0]}")
        return matches[0]
    return preferred_path  # fall through so the original error message still fires

diverse_sentences_path = resolve_path(diverse_sentences_path, "diverse_sentences*.csv")
sinhala_mixed_path = resolve_path(sinhala_mixed_path, "sinhala_mixed_dataset*.csv")

try:
    diverse_df = pd.read_csv(diverse_sentences_path).dropna()
    mixed_df = pd.read_csv(sinhala_mixed_path).dropna()
    print(f"Loaded Diverse Sentences: {len(diverse_df)} parallel pairs.")
    print(f"Loaded Mixed Script: {len(mixed_df)} sentences.")
except Exception as e:
    print(f"Data loading failed. Check your file paths. Error: {e}")
    print("\nIs the intrinsic evaluation dataset attached to this notebook?")
    print("Add it via the 'Add Input' panel on the right, or check what IS attached:")
    for p in sorted(glob.glob("/kaggle/input/*")):
        print(" -", p)

Loaded Diverse Sentences: 500 parallel pairs.
Loaded Mixed Script: 500 sentences.


Model & Tokenizer Initialization

In [4]:
# The official repository is gated and needs Llama-3 access on the token above.
# NousResearch mirrors the same weights ungated; it is only used if the official
# repository cannot be reached, and the run prints which one it took so the
# provenance stays on the record.
CANDIDATE_IDS = [
    "meta-llama/Meta-Llama-3-8B",
    "NousResearch/Meta-Llama-3-8B",
]

model = tokenizer = None
for candidate in CANDIDATE_IDS:
    try:
        print(f"Loading {candidate}...")
        tokenizer = AutoTokenizer.from_pretrained(candidate, token=HF_TOKEN)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # Load model natively in fp16 across both GPUs
        model = AutoModelForCausalLM.from_pretrained(
            candidate,
            device_map="auto",
            torch_dtype=torch.float16,
            token=HF_TOKEN,
            low_cpu_mem_usage=True
        )
        model_id = candidate
        break
    except Exception as e:
        print(f"  could not load {candidate}: {type(e).__name__}: {e}")
        model = tokenizer = None

if model is None:
    raise RuntimeError(
        "Neither Llama-3-8B repository could be loaded. Request access at "
        "https://huggingface.co/meta-llama/Meta-Llama-3-8B and check that the "
        "HF_TOKEN secret belongs to the approved account."
    )

model.eval()

# Initialize our custom evaluator
evaluator = IntrinsicEvaluator(model, tokenizer)

print(f"Model loaded from {model_id}.")
print(f"Precision: {model.dtype} | Device Map: {model.hf_device_map}")

Loading meta-llama/Meta-Llama-3-8B...
  could not load meta-llama/Meta-Llama-3-8B: OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B.
403 Client Error. (Request ID: Root=1-6aabbf99-01b6efbb06793f136b73e5e1;3d064e50-1771-4824-97d4-52bf3c5710d9)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Meta-Llama-3-8B to ask for access.
Loading NousResearch/Meta-Llama-3-8B...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

Model loaded from NousResearch/Meta-Llama-3-8B.
Precision: torch.float16 | Device Map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


Tokenizer check

Llama-3 and Llama-3.1 share one tokenizer, so this run must reproduce the
Sinhala-script fertility already recorded for Llama-3.1-8B, 9.68 tokens per
word, to within the one-token-per-sentence difference that the BOS token makes.
If it does not, the model that loaded is not the one intended and the
comparison against SinLlama would be meaningless.

The same cell prints SinLlama's extended vocabulary for contrast.

In [5]:
pieces = sum(len(tokenizer(t, add_special_tokens=False)["input_ids"])
             for t in diverse_df["sinhala_unicode"])
words = sum(len(t.split()) for t in diverse_df["sinhala_unicode"])
fertility = pieces / words

rom_pieces = sum(len(tokenizer(t, add_special_tokens=False)["input_ids"])
                 for t in diverse_df["sinhala_romanized"])
rom_words = sum(len(t.split()) for t in diverse_df["sinhala_romanized"])

print(f"vocabulary size          : {len(tokenizer):,}")
print(f"Sinhala-script fertility : {fertility:.3f} tokens/word   (expected 9.676)")
print(f"Romanized fertility      : {rom_pieces / rom_words:.3f} tokens/word   (expected 2.229)")
print()
print("For reference, SinLlama's extended vocabulary is 139,336 and scores")
print("1.436 tokens/word on the same Sinhala-script text.")

assert abs(fertility - 9.676) < 0.05, (
    f"Sinhala fertility is {fertility:.3f}, not the 9.676 the Llama-3 tokenizer "
    "gives. Wrong model or wrong corpus -- stop and check before running further."
)
print("\nTokenizer check passed.")

vocabulary size          : 128,256
Sinhala-script fertility : 9.676 tokens/word   (expected 9.676)
Romanized fertility      : 2.229 tokens/word   (expected 2.229)

For reference, SinLlama's extended vocabulary is 139,336 and scores
1.436 tokens/word on the same Sinhala-script text.

Tokenizer check passed.


Pilot Run

In [6]:
import csv

pilot_output_path = 'pilot_intrinsic_results.csv'
print("Starting Pilot Run (First 20 items)...")

with open(pilot_output_path, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'unicode_nscored', 'unicode_nchars', 'unicode_nbytes', 'unicode_nwords',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw',
        'romanized_nscored', 'romanized_nchars', 'romanized_nbytes', 'romanized_nwords'
    ])

    pilot_sample = diverse_df.head(20)

    for index, row in pilot_sample.iterrows():
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            uni_metrics['n_scored'], uni_metrics['n_chars'], uni_metrics['n_bytes'], uni_metrics['n_words'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw'],
            rom_metrics['n_scored'], rom_metrics['n_chars'], rom_metrics['n_bytes'], rom_metrics['n_words']
        ])

        if (index + 1) % 5 == 0:
            print(f"Processed {index + 1}/20 parallel pairs...")

print(f"\n Pilot run complete. Results saved to {pilot_output_path}")
preview_df = pd.read_csv(pilot_output_path)
print("\nFirst 3 rows of calculated metrics:")
display(preview_df.head(3))

Starting Pilot Run (First 20 items)...
Processed 5/20 parallel pairs...
Processed 10/20 parallel pairs...
Processed 15/20 parallel pairs...
Processed 20/20 parallel pairs...

 Pilot run complete. Results saved to pilot_intrinsic_results.csv

First 3 rows of calculated metrics:


,item_index,unicode_nll,unicode_bpb,unicode_bpc,unicode_bpw,unicode_nscored,unicode_nchars,unicode_nbytes,unicode_nwords,romanized_nll,romanized_bpb,romanized_bpc,romanized_bpw,romanized_nscored,romanized_nchars,romanized_nbytes,romanized_nwords
0,0,64.380477,1.190787,3.096047,15.480233,54,30,78,6,88.219663,3.349318,3.349318,21.212345,14,38,38,6
1,1,47.300477,1.033942,2.843340,17.060041,45,24,66,4,73.661942,3.795419,3.795419,26.567930,10,28,28,4
2,2,49.950287,0.868229,2.324614,12.010505,55,31,83,6,87.942427,3.524281,3.524281,21.145684,14,36,36,6


Full Intrinsic Evaluation

In [7]:
import csv
import pandas as pd
from tqdm import tqdm

full_diverse_output = 'Meta-Llama-3-8B_diverse_intrinsic.csv'
print(f"Starting full parallel evaluation on {len(diverse_df)} items...")

with open(full_diverse_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'unicode_nscored', 'unicode_nchars', 'unicode_nbytes', 'unicode_nwords',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw',
        'romanized_nscored', 'romanized_nchars', 'romanized_nbytes', 'romanized_nwords'
    ])

    for index, row in tqdm(diverse_df.iterrows(), total=len(diverse_df), desc="Processing Parallel Pairs"):
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            uni_metrics['n_scored'], uni_metrics['n_chars'], uni_metrics['n_bytes'], uni_metrics['n_words'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw'],
            rom_metrics['n_scored'], rom_metrics['n_chars'], rom_metrics['n_bytes'], rom_metrics['n_words']
        ])

print(f"Diverse evaluation complete. Saved to {full_diverse_output}")

full_mixed_output = 'Meta-Llama-3-8B_mixed_intrinsic.csv'
mixed_text_col = 'text' if 'text' in mixed_df.columns else mixed_df.columns[0]
print(f"\nStarting full mixed-script evaluation on {len(mixed_df)} items (column: '{mixed_text_col}')...")

with open(full_mixed_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index', 'mixed_nll', 'mixed_bpb', 'mixed_bpc', 'mixed_bpw',
        'mixed_nscored', 'mixed_nchars', 'mixed_nbytes', 'mixed_nwords'
    ])

    for index, row in tqdm(mixed_df.iterrows(), total=len(mixed_df), desc="Processing Mixed Script"):
        mix_metrics = evaluator.evaluate_sequence(row[mixed_text_col])
        writer.writerow([
            index,
            mix_metrics['nll'], mix_metrics['bpb'], mix_metrics['bpc'], mix_metrics['bpw'],
            mix_metrics['n_scored'], mix_metrics['n_chars'], mix_metrics['n_bytes'], mix_metrics['n_words']
        ])

print(f"Mixed-script evaluation complete. Saved to {full_mixed_output}")

Starting full parallel evaluation on 500 items...


Processing Parallel Pairs: 100%|██████████| 500/500 [01:27<00:00,  5.72it/s]


Diverse evaluation complete. Saved to Meta-Llama-3-8B_diverse_intrinsic.csv

Starting full mixed-script evaluation on 500 items (column: 'text')...


Processing Mixed Script: 100%|██████████| 500/500 [00:52<00:00,  9.46it/s]

Mixed-script evaluation complete. Saved to Meta-Llama-3-8B_mixed_intrinsic.csv


Pooled summary

Corpus-pooled, exactly as the paper computes it: the numerator and the
denominator are summed over the whole corpus and divided once. Averaging
per-sentence ratios would let short sentences dominate and would not reproduce
the recorded numbers.

The SinLlama column is the run already in the repository. The last block is the
comparison the paper needs, and it is the whole point of this notebook.

In [8]:
LN2 = math.log(2)

def pooled(csv_path, prefix):
    d = pd.read_csv(csv_path, encoding='utf-8-sig')
    nll = d[f'{prefix}_nll'].sum()
    return {
        'ppl': math.exp(nll / d[f'{prefix}_nscored'].sum()),
        'bpb': nll / (d[f'{prefix}_nbytes'].sum() * LN2),
        'bpc': nll / (d[f'{prefix}_nchars'].sum() * LN2),
        'bpw': nll / (d[f'{prefix}_nwords'].sum() * LN2),
        'tok_per_word': d[f'{prefix}_nscored'].sum() / d[f'{prefix}_nwords'].sum(),
    }

uni = pooled(full_diverse_output, 'unicode')
rom = pooled(full_diverse_output, 'romanized')
mix = pooled(full_mixed_output, 'mixed')

summary = pd.DataFrame(
    [uni, rom, mix],
    index=['Sinhala script', 'Romanized', 'Mixed script'],
)
print(f"Meta-Llama-3-8B, corpus-pooled ({len(diverse_df)} pairs, {len(mixed_df)} mixed)")
print(summary.to_string(float_format=lambda x: f"{x:10.4f}"))

# Values already recorded for SinLlama, which is this model plus a Sinhala
# extended tokenizer and continual pre-training.
SINLLAMA = {
    'Sinhala script': {'ppl': 74.559, 'bpb': 0.6282, 'bpw': 8.932, 'tok_per_word': 1.436},
    'Romanized':      {'ppl': 645.750, 'bpb': 3.3714, 'bpw': 20.807, 'tok_per_word': 2.229},
    'Mixed script':   {'ppl': 701.368, 'bpb': 1.3967, 'bpw': 16.727, 'tok_per_word': 1.769},
}

print()
print("=" * 74)
print("  SinLlama minus its parent, on identical text")
print("=" * 74)
print(f"{'condition':16s} {'metric':6s} {'Llama-3-8B':>12s} {'SinLlama':>12s} {'change':>12s}")
for cond, ours in (('Sinhala script', uni), ('Romanized', rom), ('Mixed script', mix)):
    for metric in ('bpb', 'bpw', 'ppl', 'tok_per_word'):
        base, sin = ours[metric], SINLLAMA[cond][metric]
        print(f"{cond:16s} {metric:6s} {base:12.4f} {sin:12.4f} {sin - base:+12.4f}")

print()
print(f"Reported perplexity degradation ratio, Romanized over Sinhala script:")
print(f"  Llama-3-8B : {rom['ppl'] / uni['ppl']:9.2f}x")
print(f"  SinLlama   : {SINLLAMA['Romanized']['ppl'] / SINLLAMA['Sinhala script']['ppl']:9.2f}x")
print(f"Real per-byte gap, Romanized minus Sinhala script:")
print(f"  Llama-3-8B : {rom['bpb'] - uni['bpb']:9.4f} bits per byte")
print(f"  SinLlama   : {SINLLAMA['Romanized']['bpb'] - SINLLAMA['Sinhala script']['bpb']:9.4f} bits per byte")

Meta-Llama-3-8B, corpus-pooled (500 pairs, 500 mixed)
                      ppl        bpb        bpc        bpw  tok_per_word
Sinhala script     3.2477     1.1566     3.0660    16.4442        9.6764
Romanized       1008.9979     3.6040     3.6140    22.2425        2.2290
Mixed script       6.3933     1.6462     3.2450    19.7167        7.3664

  SinLlama minus its parent, on identical text
condition        metric   Llama-3-8B     SinLlama       change
Sinhala script   bpb          1.1566       0.6282      -0.5284
Sinhala script   bpw         16.4442       8.9320      -7.5122
Sinhala script   ppl          3.2477      74.5590     +71.3113
Sinhala script   tok_per_word       9.6764       1.4360      -8.2404
Romanized        bpb          3.6040       3.3714      -0.2326
Romanized        bpw         22.2425      20.8070      -1.4355
Romanized        ppl       1008.9979     645.7500    -363.2479
Romanized        tok_per_word       2.2290       2.2290      +0.0000
Mixed script     bpb       